In [0]:
CREATE OR REPLACE VIEW gold.analistas.vw_producaocontagem2025 AS (

WITH ESTOQUE_LOCALIZACAO AS (
    SELECT
        CodProduto,
        MAX(CodLocal) AS CodLocalEstoque,
        MAX(NomeLocal) AS DescricaoLocalEstoque
    FROM gold.sankhya.fato_estoque
    GROUP BY CodProduto
),

CAPACIDADE_E_METAS AS (
    -- DESKTOP: Híbrido P90=5274 + picos=produção | Atualizado 2026-08-18
    -- Jan/Fev excedem P90 → cap = produção real; demais meses = P90 FLAT
    SELECT 2025 AS Ano, 1 AS Mes_Numero, 'DESKTOP' AS Linha, 'DESKTOP' AS Subgrupo, NULL AS _col5, NULL AS _col6, NULL AS _col7, 0.85 AS Meta_Capacidade, 22 AS Dias_Uteis, NULL AS _col10, 5512 AS Capacidade_Mensal_Ajustada UNION ALL
    SELECT 2025, 2, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 20, NULL, 6576 UNION ALL
    SELECT 2025, 3, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 19, NULL, 5274 UNION ALL
    SELECT 2025, 4, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 20, NULL, 5274 UNION ALL
    SELECT 2025, 5, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 21, NULL, 5274 UNION ALL
    SELECT 2025, 6, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 20, NULL, 5274 UNION ALL
    SELECT 2025, 7, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 23, NULL, 5274 UNION ALL
    SELECT 2025, 8, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 21, NULL, 5274 UNION ALL
    SELECT 2025, 9, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 22, NULL, 5274 UNION ALL
    SELECT 2025, 10, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 23, NULL, 5274 UNION ALL
    SELECT 2025, 11, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 19, NULL, 5274 UNION ALL
    SELECT 2025, 12, 'DESKTOP', 'DESKTOP', NULL, NULL, NULL, 0.85, 21, NULL, 5274

    UNION ALL

    -- TVS: capacidade modular = produção real mensal (transição Serra/Manaus 2025)
    SELECT 2025, 1, 'TVS', 'TV24/32', NULL, NULL, NULL, 0.95, 22, NULL, 8952 UNION ALL
    SELECT 2025, 1, 'TVS', 'TV43/50', NULL, NULL, NULL, 0.95, 22, NULL, 6939 UNION ALL
    SELECT 2025, 1, 'TVS', 'TV60/65', NULL, NULL, NULL, 0.95, 22, NULL, 1036 UNION ALL
    SELECT 2025, 2, 'TVS', 'TV24/32', NULL, NULL, NULL, 0.95, 20, NULL, 624 UNION ALL
    SELECT 2025, 2, 'TVS', 'TV43/50', NULL, NULL, NULL, 0.95, 20, NULL, 2001 UNION ALL
    SELECT 2025, 2, 'TVS', 'TV60/65', NULL, NULL, NULL, 0.95, 20, NULL, 482 UNION ALL
    SELECT 2025, 3, 'TVS', 'TV24/32', NULL, NULL, NULL, 0.95, 19, NULL, 200 UNION ALL
    SELECT 2025, 3, 'TVS', 'TV43/50', NULL, NULL, NULL, 0.95, 19, NULL, 99 UNION ALL
    SELECT 2025, 4, 'TVS', 'TV43/50', NULL, NULL, NULL, 0.95, 20, NULL, 1068 UNION ALL
    SELECT 2025, 5, 'TVS', 'TV43/50', NULL, NULL, NULL, 0.95, 21, NULL, 338 UNION ALL
    SELECT 2025, 5, 'TVS', 'TV60/65', NULL, NULL, NULL, 0.95, 21, NULL, 58 UNION ALL
    SELECT 2025, 8, 'TVS', 'TV43/50', NULL, NULL, NULL, 0.95, 21, NULL, 180

    UNION ALL

    -- MONITORES: Híbrido P90=10109 + pico Set=produção | Jan-Abr = produção real (ramp-up) | Atualizado 2026-08-18
    SELECT 2025, 1, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 22, NULL, 2807 UNION ALL
    SELECT 2025, 2, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 20, NULL, 206 UNION ALL
    SELECT 2025, 3, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 19, NULL, 30 UNION ALL
    SELECT 2025, 4, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 20, NULL, 667 UNION ALL
    SELECT 2025, 5, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 21, NULL, 10109 UNION ALL
    SELECT 2025, 6, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 20, NULL, 10109 UNION ALL
    SELECT 2025, 7, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 23, NULL, 10109 UNION ALL
    SELECT 2025, 8, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 21, NULL, 10109 UNION ALL
    SELECT 2025, 9, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 22, NULL, 12068 UNION ALL
    SELECT 2025, 10, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 23, NULL, 10109 UNION ALL
    SELECT 2025, 11, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 19, NULL, 10109 UNION ALL
    SELECT 2025, 12, 'MONITORES', 'MONITORES', NULL, NULL, NULL, 0.98, 21, NULL, 10109
),

-- 🛠️ CORREÇÃO CRÍTICA: Trocamos o INNER JOIN por LEFT JOIN e simplificamos a data
BASE_PRODUCAO_SLA_2025 AS (
    -- Grão: (OP, Produto, Data) — cada dia de apontamento vira uma linha própria
    -- Corrige divergência vs PCP que conta pela data real de cada bipagem
    SELECT
        ap.OrdemProducao,
        COALESCE(pa.CodProdutoAcabado, 0)                                                   AS CodProd,
        MAX(pr.NumUnicoNotaPedido)                                                         AS NumUnicoNotaPedido,
        YEAR(CAST(COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) AS DATE))           AS Ano,
        MONTH(CAST(COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) AS DATE))          AS Mes,
        WEEKOFYEAR(CAST(COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) AS DATE))     AS Semana,
        CAST(COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) AS DATE)                 AS Data_Producao,
        MIN(COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento))                          AS DataHora_Apontamento,
        COUNT(CASE WHEN ap.DataHoraEmbalagem IS NOT NULL THEN ap.SerieProdutoAcabado END) AS Quantidade_Produzida
    FROM gold.sankhya.fato_ordem_producao_seriepa_ciclo ap
    LEFT JOIN gold.sankhya.fato_ordem_producao pr 
        ON pr.OrdemProducao = ap.OrdemProducao
    LEFT JOIN gold.sankhya.fato_ordem_producao_seriepa pa 
        ON ap.SerieProdutoAcabado = pa.SerieProdutoAcabado
    WHERE COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) >= '2025-01-01'
      AND COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) < '2026-01-01'
    GROUP BY ap.OrdemProducao, pa.CodProdutoAcabado, CAST(COALESCE(ap.DataHoraEmbalagem, ap.DataHoraApontamento) AS DATE)
),

SKU_ATRIBUTOS_2025 AS (
    SELECT
        P.CodProduto, P.DescricaoProduto, P.Marca AS Fornecedor, P.ModeloMkt AS Modelo,
        GP.NomeGrupoPai AS Familia, GP.LinhaDeNegocio, GP.NomeGrupoFamilia, P.UsadoComo
    FROM gold.sankhya.dim_produtos P
    INNER JOIN gold.sankhya.dim_grupo_produtos GP ON GP.CodGrupoProduto = P.CodGrupoProduto
),

PLANEJAMENTO_OP_2025 AS (
    SELECT
        OrdemProducao, CodProdutoAcabado, COUNT(DISTINCT SerieProdutoAcabado) AS Qtd_Planejada_Item
    FROM gold.sankhya.fato_ordem_producao_seriepa
    GROUP BY OrdemProducao, CodProdutoAcabado
),

REPAROS_2025 AS (
    -- Agrega dados de reparo/sintoma por (OP, Produto, Data_Producao) para casar com o grão da produção
    SELECT
        fs.OrdemProducao,
        pa.CodProdutoAcabado                                                                AS CodProd_Reparo,
        CAST(COALESCE(ciclo.DataHoraEmbalagem, ciclo.DataHoraApontamento) AS DATE)           AS Data_Producao_Reparo,
        COUNT(*)                                                                             AS Qtd_Reparos,
        COUNT(CASE WHEN fs.CodigProdutoBelMicro > 0 THEN 1 END)                             AS Pecas_Substituidas,
        ARRAY_JOIN(COLLECT_SET(dr.Descricao), ' | ')                                         AS Sintomas,
        ARRAY_JOIN(COLLECT_SET(dat.Defeito), ' | ')                                          AS Defeitos,
        ARRAY_JOIN(COLLECT_SET(aat.Acao), ' | ')                                             AS Acoes,
        ARRAY_JOIN(COLLECT_SET(CASE WHEN fs.CodigProdutoBelMicro > 0 THEN CAST(fs.CodigProdutoBelMicro AS STRING) END), ' | ') AS SKU_Componente_Trocado,
        ARRAY_JOIN(COLLECT_SET(CASE WHEN fs.CodigProdutoBelMicro > 0 THEN comp.DescricaoProduto END), ' | ') AS Descricao_Componente_Trocado,
        ARRAY_JOIN(COLLECT_SET(CAST(fs.DataAlteracaoReparo AS STRING)), ' | ')                AS Datas_Conclusao_Reparo,
        ARRAY_JOIN(COLLECT_SET(fs.SerieProdutoAcabado), ' | ')                               AS Serie_PA,
        ARRAY_JOIN(COLLECT_SET(fs.Origem || ' - LINHA ' || CAST(fs.LinhaDoApontamento AS STRING)), ' | ') AS Centro_Trabalho_Reparo
    FROM gold.sankhya.fato_sintoma_serie fs
    INNER JOIN gold.sankhya.fato_ordem_producao_seriepa pa
        ON fs.SerieProdutoAcabado = pa.SerieProdutoAcabado
    INNER JOIN gold.sankhya.fato_ordem_producao_seriepa_ciclo ciclo
        ON fs.SerieProdutoAcabado = ciclo.SerieProdutoAcabado
    LEFT JOIN gold.sankhya.dim_defeito_runin dr
        ON fs.NumeroDefeito = dr.NumeroDefeito
    LEFT JOIN gold.sankhya.dim_defeito_assistencia_tecnica dat
        ON fs.CodDefeito = dat.CodDefeito
    LEFT JOIN gold.sankhya.dim_acao_assistencia_tecnica aat
        ON fs.CodAcao = aat.CodAcao
    LEFT JOIN gold.sankhya.dim_produtos comp
        ON fs.CodigProdutoBelMicro = comp.CodProduto
    WHERE fs.DataApontamento >= '2025-01-01' AND fs.DataApontamento < '2026-01-01'
    GROUP BY fs.OrdemProducao, pa.CodProdutoAcabado, CAST(COALESCE(ciclo.DataHoraEmbalagem, ciclo.DataHoraApontamento) AS DATE)
)

SELECT
    EST.CodLocalEstoque,
    EST.DescricaoLocalEstoque,
    COALESCE(METAS.Meta_Capacidade, 0.95)            AS Meta_Capacidade,
    CASE WHEN ROW_NUMBER() OVER (PARTITION BY CONSULTA.Ano, CONSULTA.Mes, METAS.Subgrupo ORDER BY CONSULTA.DataProducao, CONSULTA.OP) = 1
         THEN COALESCE(METAS.Capacidade_Mensal_Ajustada, 0) ELSE 0 END AS Cap_Mensal_Ajustada_Subgrupo,
    CASE WHEN ROW_NUMBER() OVER (PARTITION BY CONSULTA.Ano, CONSULTA.Mes, METAS.Subgrupo ORDER BY CONSULTA.DataProducao, CONSULTA.OP) = 1
         THEN COALESCE(METAS.Capacidade_Mensal_Ajustada, 0) ELSE 0 END AS Cap_Turno_8h_Subgrupo,
    COALESCE(METAS.Dias_Uteis, 0)                 AS Dias_Uteis_Mes,
    CONSULTA.*
FROM (
    SELECT
        'CONTAGEM/MG'                                               AS Planta,
        COALESCE(pp.DescricaoProcesso, 'HISTORICO 2025')            AS Processo,
        PROD.OrdemProducao                                          AS OP,
        PROD.NumUnicoNotaPedido                                     AS NumUnicoNota,
        O.NumNota                                                   AS NumeroNota,
        PROD.Ano, PROD.Mes, PROD.Semana, PROD.Data_Producao        AS DataProducao,
        PROD.DataHora_Apontamento                                   AS DataHoraApontamento,
        
        CASE MONTH(PROD.DataHora_Apontamento)
            WHEN 1  THEN '01. JANEIRO' WHEN 2  THEN '02. FEVEREIRO' WHEN 3  THEN '03. MARÇO'
            WHEN 4  THEN '04. ABRIL'   WHEN 5  THEN '05. MAIO'      WHEN 6  THEN '06. JUNHO'
            WHEN 7  THEN '07. JULHO'   WHEN 8  THEN '08. AGOSTO'    WHEN 9  THEN '09. SETEMBRO'
            WHEN 10 THEN '10. OUTUBRO' WHEN 11 THEN '11. NOVEMBRO'  WHEN 12 THEN '12. DEZEMBRO'
        END AS Mes_Ordenado,

        PROD.CodProd, 
        COALESCE(SKU.DescricaoProduto, 'PRODUTO HISTORICO 2025')   AS DescricaoProduto, 
        COALESCE(SKU.Fornecedor, 'N/A')                             AS Fornecedor, 
        COALESCE(SKU.Modelo, 'N/A')                                 AS Modelo,
        COALESCE(SKU.Familia, 'N/A')                                AS Familia, 
        COALESCE(SKU.LinhaDeNegocio, 'N/A')                         AS LinhaDeNegocio, 
        COALESCE(SKU.NomeGrupoFamilia, 'N/A')                       AS NomeGrupoFamilia, 
        COALESCE(SKU.UsadoComo, 'N/A')                              AS UsadoComo,

        CASE
            WHEN UPPER(COALESCE(SKU.Familia, '')) IN ('MONITORES', 'MONITOR') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '49("| |POL|INCH)' THEN '49"'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(32|34)("| |POL|INCH)' THEN REGEXP_EXTRACT(UPPER(SKU.DescricaoProduto), '(32|34)', 1) || '"'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '27("| |POL|INCH)' THEN '27"'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(19|19\.5|20|21|21\.5|22|23\.8|24)("| |POL|INCH)?' THEN REGEXP_EXTRACT(UPPER(SKU.DescricaoProduto), '(19\.5|21\.5|23\.8|19|20|21|22|24)', 1) || '"'
                    ELSE 'Indefinida'
                END
            WHEN UPPER(COALESCE(SKU.Familia, '')) IN ('TVS', 'TV', 'TELEVISORES') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(24|32|43|50|55|60|65|70|75)("| |POL|INCH)?' THEN REGEXP_EXTRACT(UPPER(SKU.DescricaoProduto), '(24|32|43|50|55|60|65|70|75)', 1) || '"'
                    ELSE 'Indefinida'
                END
            ELSE 'N/A'
        END AS Polegada_Estimada,

        CASE
            WHEN UPPER(COALESCE(SKU.Familia, '')) IN ('MONITORES', 'MONITOR') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '49("| |POL|INCH)' THEN 'M49'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(32|34)("| |POL|INCH)' THEN 'M32/34'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '27("| |POL|INCH)' THEN 'M27'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(19|19\.5|20|21|21\.5|22|23\.8|24)("| |POL|INCH)?' THEN 'M19/21/24'
                    ELSE 'MONITOR - VERIFICAR CADASTRO'
                END
            WHEN UPPER(COALESCE(SKU.Familia, '')) IN ('TVS', 'TV', 'TELEVISORES') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(24|32)("| |POL|INCH)?' THEN 'TV24/32'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(43|50)("| |POL|INCH)?' THEN 'TV43/50'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(55|60|65|70|75)("| |POL|INCH)?' THEN 'TV60/65'
                    ELSE 'TV - VERIFICAR CADASTRO'
                END
            ELSE 'OUTROS'
        END AS Subgrupo_TV_Monitor,

        CASE
            WHEN UPPER(COALESCE(SKU.DescricaoProduto, '')) RLIKE '(AIO|ALL IN ONE|ALL-IN-ONE)' THEN 'AIO'
            WHEN UPPER(COALESCE(SKU.Familia, '')) IN ('DESKTOP', 'DESKTOPS', 'COMPUTADORES') OR UPPER(COALESCE(SKU.Familia, '')) LIKE '%DESKTOP%' THEN
                CASE
                    WHEN UPPER(COALESCE(SKU.DescricaoProduto, '')) LIKE '%GAMER%' THEN 'GAMER'
                    WHEN UPPER(COALESCE(SKU.DescricaoProduto, '')) RLIKE '(RTX|RX ?5[87]0|RX ?6[0-9]|RX ?7[0-9]|GTX ?16|GTX ?10[67]0)' THEN 'GAMER'
                    WHEN UPPER(COALESCE(SKU.DescricaoProduto, '')) LIKE '%SLIM%' THEN 'SLIM'
                    ELSE 'DESK'
                END
            ELSE 'OUTROS'
        END AS SubgrupoProduto,

        PROD.Quantidade_Produzida                                   AS Qtd_Produzida,
        -- Distribuição proporcional: cada dia recebe fração da planejada proporcional à sua produção
        ROUND(PROD.Quantidade_Produzida * COALESCE(PLAN_OP.Qtd_Planejada_Item, 0)
            / NULLIF(SUM(PROD.Quantidade_Produzida) OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd), 0)
        ) AS Qtd_Planejada,
        ROUND(PROD.Quantidade_Produzida * (COALESCE(PLAN_OP.Qtd_Planejada_Item, 0) - SUM(PROD.Quantidade_Produzida) OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd))
            / NULLIF(SUM(PROD.Quantidade_Produzida) OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd), 0)
        ) AS Saldo,
        'PRODUZIDO'                                                 AS Situacao,
        CAST(O.DataNegociacao AS DATE)                              AS DataFaturamento,
        
        0 AS Separacao, 0 AS Montagem, 0 AS Qualidade, 0 AS Runin, 0 AS Embalagem,
        
        COALESCE(REP.Qtd_Reparos, 0) AS Qtd_Reparos,
        COALESCE(REP.Pecas_Substituidas, 0) AS Pecas_Substituidas,
        COALESCE(REP.Sintomas, 'Sem Registro') AS Sintomas,
        COALESCE(REP.Defeitos, 'Sem Registro') AS Defeitos,
        COALESCE(REP.Acoes, 'Sem Registro') AS Acoes,
        COALESCE(REP.SKU_Componente_Trocado, 'Nenhuma') AS SKU_Componente_Trocado,
        COALESCE(REP.Descricao_Componente_Trocado, 'Nenhuma') AS Descricao_Componente_Trocado,
        COALESCE(REP.Datas_Conclusao_Reparo, 'Nenhum') AS Datas_Conclusao_Reparo,
        COALESCE(REP.Serie_PA, 'Nenhum') AS Serie_PA,
        COALESCE(REP.Centro_Trabalho_Reparo, 'Nenhum') AS Centro_Trabalho_Reparo
    FROM BASE_PRODUCAO_SLA_2025 PROD
    LEFT JOIN SKU_ATRIBUTOS_2025 SKU ON PROD.CodProd = SKU.CodProduto
    LEFT JOIN gold.sankhya.fato_ordem_producao fop ON PROD.OrdemProducao = fop.OrdemProducao
    LEFT JOIN gold.sankhya.dim_processo_producao pp ON fop.CodProcessoUnico = pp.CodProcessoUnico
    LEFT JOIN PLANEJAMENTO_OP_2025 PLAN_OP ON PROD.OrdemProducao = PLAN_OP.OrdemProducao AND PROD.CodProd = PLAN_OP.CodProdutoAcabado
    LEFT JOIN gold.sankhya.fato_operacoes O ON PROD.NumUnicoNotaPedido = O.NumUnicoNota
    LEFT JOIN REPAROS_2025 REP ON PROD.OrdemProducao = REP.OrdemProducao AND PROD.CodProd = REP.CodProd_Reparo AND PROD.Data_Producao = REP.Data_Producao_Reparo
) AS CONSULTA
LEFT JOIN ESTOQUE_LOCALIZACAO EST ON CONSULTA.CodProd = EST.CodProduto
LEFT JOIN CAPACIDADE_E_METAS METAS ON CONSULTA.Ano = METAS.Ano AND CONSULTA.Mes = METAS.Mes_Numero AND (
    CASE
        WHEN CONSULTA.SubgrupoProduto = 'AIO' THEN 'DESKTOP'
        WHEN CONSULTA.Subgrupo_TV_Monitor IN ('M19/21/24', 'M27', 'M32/34', 'M49') THEN 'MONITORES'
        WHEN CONSULTA.SubgrupoProduto IN ('DESK', 'GAMER', 'SLIM') THEN 'DESKTOP'
        WHEN CONSULTA.Subgrupo_TV_Monitor IN ('TV24/32', 'TV43/50', 'TV60/65') THEN CONSULTA.Subgrupo_TV_Monitor
        ELSE NULL
    END
) = METAS.Subgrupo
WHERE CONSULTA.Familia <> 'N/A'
  AND CONSULTA.Familia <> 'COMPONENTES'
);